# Final model: RT-DETR-L - walkthrough and reproduction

This notebook shows the full pipeline of the final model step by step on the mini dataset (CPU, ~5 min), then how the same code is run on Kaggle GPU for the real result (val mAP 0.252, LB 0.340 public / 0.270 private).

Everything is imported from the repo; nothing is reimplemented here.

## 1. Code
Clone the repo and install the pinned environment (`code/requirements.txt` = CPU torch + ultralytics 8.4.214 + pycocotools). On Kaggle only `ultralytics==8.4.214 pycocotools` are installed on top of the preinstalled CUDA torch.

In [1]:
!git clone --depth 1 https://github.com/BodiyaKol/ucu-ai-hm-2-object-detection_Kolodchak.git repo
%cd repo/code
!pip install -q -r requirements.txt

fatal: destination path 'repo' already exists and is not an empty directory.
/content/repo/code


## 2. Data
The mini dataset (2 clips x 20 frames per split, exact competition layout) that every script was tested on: https://drive.google.com/file/d/19WWHDg0Y7udzrkwkiTAnCoWZj5bTU5To/view (`mini.zip`). The cell below downloads it into `code/data/mini/`. On Kaggle the full competition data is found automatically by `src.config.get_data_root`.

In [13]:
!pip install -q gdown
!gdown --id 19WWHDg0Y7udzrkwkiTAnCoWZj5bTU5To -O mini.zip
!mkdir -p data && unzip -qo mini.zip -d data && rm mini.zip
import sys; sys.path.insert(0, 'scripts')
from src.config import load_config, get_data_root
cfg = load_config('configs/protocol.yaml')
root = get_data_root(cfg)
print('data root:', root)
!ls {root}

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=19WWHDg0Y7udzrkwkiTAnCoWZj5bTU5To
From (redirected): https://drive.google.com/uc?id=19WWHDg0Y7udzrkwkiTAnCoWZj5bTU5To&confirm=t&uuid=040510c1-8e06-4b88-ad9e-76076bd07784
To: /content/repo/code/mini.zip

data root: /content/repo/code/data/mini
test  test_images.csv  train  val


## 3. Preparation (`scripts/prepare_data.py`, M1.5)
Every 10th train frame, ignore regions painted gray (114), boxes covered >50% by ignore dropped, YOLO labels + `train.txt`, COCO json for val with ignore regions as `iscrowd=1`. Output: `outputs/prepared/`. On Kaggle this is the `hw2-01-prepare` kernel, attached to training kernels as `kernel_sources`.

In [14]:
!python scripts/prepare_data.py
!cat outputs/prepared/prepare_stats.json

data root: /content/repo/code/data/mini
output: /content/repo/code/outputs/prepared
frame_step: 10


{
  "train_frames": 4,
  "frames_masked": 4,
  "boxes_kept": 84,
  "boxes_dropped_in_ignore": 6,
  "train_coco_objects": 84,
  "train_coco_crowd": 96,
  "val_coco_objects": 1616,
  "val_coco_crowd": 448,
  "frame_step": 10,
  "val_frames": 40,
  "data_root": "/content/repo/code/data/mini"
}
data.yaml: /content/repo/code/outputs/prepared/data.yaml
{
  "train_frames": 4,
  "frames_masked": 4,
  "boxes_kept": 84,
  "boxes_dropped_in_ignore": 6,
  "train_coco_objects": 84,
  "train_coco_crowd": 96,
  "val_coco_objects": 1616,
  "val_coco_crowd": 448,
  "frame_step": 10,
  "val_frames": 40,
  "data_root": "/content/repo/code/data/mini"
}

## 4. Config
Shared protocol (`configs/protocol.yaml`: seed 42, frame_step 10, imgsz 1024, 500 detections, score 0.001) + model config (`configs/rtdetr/base.yaml`: `rtdetr-l.pt`, batch 4, time cap 0.75 h). `mini:` block overrides for the CPU dry-run.

In [15]:
!cat configs/protocol.yaml
!cat configs/rtdetr/base.yaml

# Shared protocol for ALL experiments (kept fixed across model families M3-M6).
# Every value below is justified by results/eda.json and results/eda_choices.json (M1.2-M1.3).
seed: 42

# ---------------------------------------------------------------- data
competition: multiple-object-video-detection-hw-2-ucu-ai-2026
local_dataset: mini            # local CPU debugging uses data/mini (exact copy of competition layout)
data_dirs:
  mini: data/mini
  annotations: data/annotations
num_classes: 8
class_names: [person, bicycle, car, van, truck, tricycle, bus, motor]

# ---------------------------------------------------------------- split (by sequence, no frame-level leakage)
train_split: train             # 56 clips
val_split: val                 # 7 clips, fixed hold-out for all experiments

# ---------------------------------------------------------------- sampling
# Consecutive frames are near-duplicates: at step 1 only 0.45% of objects are new and the same
# object keeps IoU 0.85; at s

## 5. Training + evaluation + submission (`scripts/train_yolo.py`)
One script, same for YOLO and RT-DETR (class picked from the weights name):
1. `write_data_yaml` -> Ultralytics data.yaml (train = prepared, val = original val images)
2. `RTDETR(...).train(...)` with the config values; `time` caps the run
3. `predict`: images passed as a .txt list, batch 1 (a Python list would be decoded into RAM as one batch -> OOM)
4. `coco_evaluate` (`src/eval/coco_eval.py`): pycocotools, top-500, ignore regions as crowd, per-class AP
5. `predictions_to_csv` + `check_submission` (`src/submission.py`): all 3811 test rows, top-500 by score

Mini run below: 1 epoch, 320 px, CPU.

In [ ]:
!python scripts/train_yolo.py --exp nb_mini --mini --config configs/rtdetr/base.yaml

data root: /content/repo/code/data/mini
prepared: /content/repo/code/outputs/prepared
output: /content/repo/code/outputs/nb_mini
device: cpu
settings: {'imgsz': 320, 'epochs': 1, 'batch': 2, 'workers': 0, 'close_mosaic': 2, 'time': None}
[mem] start: process 0.3 GB, available 12.2 / 13.6 GB
New https://pypi.org/project/ultralytics/8.4.215 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.214 🚀 Python-3.13.18 torch-2.15.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/repo/code/outputs/nb_mini/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic

In [12]:
import json
m = json.load(open('outputs/nb_mini/metrics.json'))
print(json.dumps({k: m[k] for k in ['exp', 'weights', 'imgsz', 'epochs', 'val', 'submission']}, indent=2))
!head -c 300 outputs/nb_mini/submission.csv

{
  "exp": "nb_mini",
  "weights": "rtdetr-l.pt",
  "imgsz": 320,
  "epochs": 1,
  "val": {
    "mAP": 0.03682947417697242,
    "AP50": 0.09631359669365082,
    "AP75": 0.02286506606709611,
    "AP_small": 0.013861386138613862,
    "AP_medium": 0.08095504711732182,
    "AP_large": 0.030803737012414848,
    "per_class_AP": {
      "person": 0.060701388675269315,
      "bicycle": null,
      "car": 0.12342631787970394,
      "van": 0.0,
      "truck": null,
      "tricycle": 0.0,
      "bus": null,
      "motor": 1.966432988879821e-05
    }
  },
  "submission": {
    "rows": 40,
    "detections": 12000,
    "empty_rows": 0
  }
}
image_id,PredictionString
clip_001_0000001,0 0.5294 705.0 765.9 37.8 112.7 0 0.5039 1700.6 793.1 39.5 145.1 0 0.4809 816.7 629.7 29.1 74.3 0 0.4299 779.1 884.4 57.3 187.1 3 0.3910 9.3 13.5 880.9 1053.6 3 0.3867 781.8 893.1 61.5 118.5 3 0.3787 13.2 5.1 978.7 1070.2 3 0.3785 3.6 1.7 1003.0 1076.2 3 0

## 6. Real run on Kaggle GPU
The same script is launched by the kernel folder `notebooks/06_rtdetr/` (`run.py` clones the repo and runs training then inference in two processes; `kernel-metadata.json` attaches the competition data and the `hw2-01-prepare` output). From `code/`:

```bash
kaggle kernels push -p notebooks/06_rtdetr --accelerator NvidiaTeslaT4
kaggle kernels status bohdankolodchak/hw2-06-rtdetr
kaggle kernels output bohdankolodchak/hw2-06-rtdetr -p outputs/06 --file-pattern "metrics.json|submission.csv"
kaggle competitions submit -c multiple-object-video-detection-hw-2-ucu-ai-2026 -f outputs/06/m6_rtdetr/submission.csv -m "RT-DETR-L 1024"
```

Test inference from saved weights (`notebooks/08b_late_rtdetr/`) attaches the finished kernel as `kernel_sources` and runs `train_yolo.py --weights .../last.pt`.

Seeds: 42 everywhere (`set_seed`), Ultralytics `deterministic=True`. Results of all runs: `code/results/*.json`, `code/experiments.md`.